# Fire & Smoke Detection — Completed Experiment

**YOLO Multi-Task Computer Vision | Task 1**

Trained YOLO26n on the cleaned Roboflow Fire & Smoke v14 dataset (5,596 images, two classes). Completed 50 epochs, held-out test evaluation, image inference, and video inference. Test mAP@50: **0.6302**; mAP@50–95: **0.2800**. See the results section and repository README for limitations and artifact locations.

**Execution note:** this notebook preserves the original Colab experimental workflow, including environment- and GPU-dependent cells. Review prerequisites and run selected stages deliberately; it has not been verified as a fully sequential CPU execution.


## Using this notebook

1. Open in Google Colab and mount the private Google Drive artifacts.
2. Clone the matching repository when `src/` helpers are required; install dependencies in a fresh runtime before imports.
3. The training and evaluation cells require a compatible GPU and the reviewed dataset; **do not rerun them merely to inspect results**.
4. The final section reads existing `best.pt` and `test_metrics.json` without training or model loading. The original dataset correction decisions remain in private Drive evidence.
5. Video and images may have third-party licensing restrictions: do not redistribute raw media or checkpoint artifacts by default.


In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess
import zipfile
import json

drive.mount("/content/drive")

ROOT = Path("/content/yolo-multi-task-computer-vision")
BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14"
)

# Restore project code if the runtime is new
if not (ROOT / ".git").exists():
    assert not ROOT.exists(), "Inspect existing project folder first"
    subprocess.run([
        "git", "clone",
        "https://github.com/Ibraheim11/yolo-multi-task-computer-vision.git",
        str(ROOT)
    ], check=True)
else:
    subprocess.run(
        ["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"],
        check=True
    )

# Check saved work
backup = BASE / "fire_smoke_dataset_backup.zip"
progress_file = BASE / "PROJECT_PROGRESS.json"
coco = BASE / "fire and smoke detector.v14i.coco.zip"

print("Dataset backup:", backup.is_file())
print("Progress saved:", progress_file.is_file())

if progress_file.is_file():
    progress = json.loads(progress_file.read_text())
    print("Review folder:", Path(progress["review_dir"]).is_dir())

print("COCO exists:", coco.is_file())

if coco.is_file():
    print("COCO size MB:", round(coco.stat().st_size / 1024**2, 2))
    print("COCO valid ZIP:", zipfile.is_zipfile(coco))

    if zipfile.is_zipfile(coco):
        with zipfile.ZipFile(coco) as z:
            print("Corrupt member:", z.testzip())

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Searching Google Drive for COCO ZIP...\n")

matches = [
    p for p in drive_root.rglob("*")
    if p.is_file()
    and p.suffix.lower() == ".zip"
    and ("coco" in p.name.lower() or "smoke" in p.name.lower())
]

for p in matches:
    print("File:", p)
    print("Size MB:", round(p.stat().st_size / 1024**2, 2))
    print("-" * 40)

print("Total matches:", len(matches))

In [ ]:
%pip install "roboflow>=1.1,<2"

In [ ]:
from pathlib import Path
from getpass import getpass
from roboflow import Roboflow

COCO_DIR = Path("/content/fire_smoke_v14_coco_source")

# Don't overwrite an existing dataset
assert not COCO_DIR.exists(), "Folder exists; inspect it first"

# Read API Key securely from Colab Secrets
from google.colab import userdata

key = userdata.get("ROBOFLOW_API_KEY")
if not key:
    key = getpass("Roboflow API Key: ")

rf = Roboflow(api_key=key)

dataset = (
    rf.workspace("gfo-fc1de")
      .project("fire-and-smoke-detector-yykya")
      .version(14)
      .download("coco", location=str(COCO_DIR))
)

print("COCO downloaded:", dataset.location)

for p in COCO_DIR.rglob("*.json"):
    print(p.relative_to(COCO_DIR))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys

ROOT = Path("/content/yolo-multi-task-computer-vision")
BASE = Path("/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14")
DATASET_DIR = ROOT / "datasets/fire_smoke_v14"
COCO_DIR = Path("/content/fire_smoke_v14_coco_source")

sys.path.insert(0, str(ROOT))

# Restore original YOLO dataset if needed
if not (DATASET_DIR / "data.yaml").is_file():
    import shutil
    backup = BASE / "fire_smoke_dataset_backup.zip"
    assert backup.is_file()
    shutil.unpack_archive(str(backup), str(ROOT / "datasets"))

# Reuse existing review decisions
progress = json.loads(
    (BASE / "PROJECT_PROGRESS.json").read_text(encoding="utf-8")
)
REVIEW_DIR = Path(progress["review_dir"])

assert (REVIEW_DIR / "review.json").is_file()
assert (REVIEW_DIR / "decisions.json").is_file()
assert len(list(COCO_DIR.rglob("*_annotations*.coco.json"))) == 3

from src.coco_provenance import verify_coco
from src.fire_smoke import sha256

decision_hash = sha256(REVIEW_DIR / "decisions.json")

REPORT = BASE / (
    "provenance_" +
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
) / "report.json"

report = verify_coco(
    DATASET_DIR,
    REVIEW_DIR,
    COCO_DIR,
    REPORT,
    source_note=(
        "Roboflow workspace gfo-fc1de, project "
        "fire-and-smoke-detector-yykya, version 14, "
        "COCO export downloaded directly in Colab."
    ),
)

assert sha256(REVIEW_DIR / "decisions.json") == decision_hash

for row in report["items"]:
    print(
        row["review_index"],
        row["id"],
        row["status"]
    )

print("\nAnnotation 6:")
print(json.dumps(report["items"][5], indent=2))

print("\nReport saved:", REPORT)

In [ ]:
from pathlib import Path
import json

report = json.loads(
    REPORT.read_text(encoding="utf-8")
)

for item in report["items"]:
    p = item["proposal"]
    xs = [point[0] for point in p["points"]]
    ys = [point[1] for point in p["points"]]

    print(f"\nAnnotation {item['review_index']}/7")

    matches = []

    for candidate in item.get("candidates", []):
        width = candidate["image"]["width"]
        height = candidate["image"]["height"]

        expected = [
            min(xs) * width,
            min(ys) * height,
            (max(xs) - min(xs)) * width,
            (max(ys) - min(ys)) * height
        ]

        for ann in candidate.get("annotations", []):
            if ann.get("whole_polygon_match") is True:
                source = ann["source"]
                actual = source["bbox"]

                differences = [
                    abs(a - b)
                    for a, b in zip(actual, expected)
                ]

                matches.append({
                    "category": source["category_name"],
                    "max_bbox_difference_px": round(max(differences), 4),
                    "segmentation_kind": ann["segmentation_kind"]
                })

    print("Whole polygon matches:", len(matches))
    for m in matches:
        print(m)

In [ ]:
import json

decision_path = REVIEW_DIR / "decisions.json"
decisions = json.loads(decision_path.read_text(encoding="utf-8"))

assert len(decisions["items"]) == 7
assert len(report["items"]) == 7

evidence_by_id = {
    item["id"]: item
    for item in report["items"]
}

for decision in decisions["items"]:
    evidence = evidence_by_id[decision["id"]]

    matches = [
        ann
        for candidate in evidence.get("candidates", [])
        for ann in candidate.get("annotations", [])
        if ann.get("whole_polygon_match") is True
        and ann.get("category_match") is True
    ]

    assert len(matches) == 1, (
        f"Expected exactly one polygon match: {decision['id']}"
    )

    decision.update(
        confirmed_format="polygon",
        format_evidence=(
            "Independent Roboflow v14 COCO export; "
            "exact image identity and matching whole polygon. "
            f"Provenance report: {REPORT}"
        ),
        class_confirmed=True,
        matches_visible_object=True,
        reason=(
            "Previously visually reviewed. "
            "Independent COCO export confirms matching polygon "
            "and class. Axis-aligned detection conversion approved."
        ),
        approve=True
    )

decision_path.write_text(
    json.dumps(decisions, indent=2),
    encoding="utf-8"
)

print("Approved:", sum(d["approve"] for d in decisions["items"]))
print("Decisions saved:", decision_path)

In [ ]:
from pathlib import Path
import json
from src.polygon_review import write_reviewed_plan

# Verify all seven approvals
decisions = json.loads(
    (REVIEW_DIR / "decisions.json").read_text(encoding="utf-8")
)

approved = sum(
    d.get("approve") is True
    for d in decisions["items"]
)

print("Approved annotations:", approved)

assert len(decisions["items"]) == 7
assert approved == 7, "Some annotations are not approved"

# Generate correction plan
PLAN = write_reviewed_plan(DATASET_DIR, REVIEW_DIR)

print("\nCorrection plan created successfully!")
print("Plan path:", PLAN)

plan = json.loads(PLAN.read_text(encoding="utf-8"))

print("\nPlan contents:")
print(json.dumps(plan, indent=2)[:12000])

In [ ]:
import json
from src.label_diagnostics import create_reviewed_copy, diagnose_labels, write_diagnosis
from src.fire_smoke import audit_dataset

# New dataset saved permanently to Google Drive
CLEAN = REVIEW_DIR / "cleaned_detection_dataset"

assert PLAN.is_file(), "Correction plan not found"
assert DATASET_DIR.is_dir(), "Original dataset not found"
assert not CLEAN.exists(), "Clean dataset already exists; inspect it first"

# Create corrected dataset copy
manifest = create_reviewed_copy(
    DATASET_DIR,
    CLEAN,
    PLAN
)

print("Clean dataset created:", CLEAN)

# Independent full dataset audit
clean_audit = audit_dataset(CLEAN)

(REVIEW_DIR / "clean_audit.json").write_text(
    json.dumps(clean_audit, indent=2),
    encoding="utf-8"
)

print("\nFINAL AUDIT RESULTS")
print("Manifest ready:", manifest["ready"])
print("Audit ready:", clean_audit["ready"])
print("Total issues:", len(clean_audit["issues"]))

for split, info in clean_audit["splits"].items():
    print(
        split,
        "Images:", info["images"],
        "Invalid labels:", info["invalid_labels"]
    )

# Final checks
assert manifest["ready"] is True
assert clean_audit["ready"] is True
assert clean_audit["issues"] == []
assert clean_audit["names"] == ["fire", "smoke"]

assert {
    split: info["images"]
    for split, info in clean_audit["splits"].items()
} == {
    "train": 3940,
    "val": 1017,
    "test": 639
}

print("\nCLEAN DATASET VALIDATION: PASS")
print("Ready for YOLO training!")

In [ ]:
from pathlib import Path
import shutil
import sys
import torch

ROOT = Path("/content/yolo-multi-task-computer-vision")
sys.path.insert(0, str(ROOT))

CLEAN = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/"
    "fire_smoke_v14/"
    "polygon_review_20261110T141343824922Z/"
    "cleaned_detection_dataset"
)

LOCAL_DATA = ROOT / "datasets/fire_smoke_v14_training"
OUTPUT = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/"
    "fire_smoke_v14/training"
)

assert torch.cuda.is_available(), "Please enable T4 GPU first"
assert (CLEAN / "data.yaml").is_file()

if not LOCAL_DATA.exists():
    shutil.copytree(CLEAN, LOCAL_DATA)

from src.fire_smoke import audit_dataset, write_training_yaml

audit = audit_dataset(LOCAL_DATA)

assert audit["ready"] is True
assert len(audit["issues"]) == 0

OUTPUT.mkdir(parents=True, exist_ok=True)

DATA_YAML = write_training_yaml(
    audit,
    OUTPUT / "data.training.yaml"
)

print("GPU:", torch.cuda.get_device_name(0))
print("Dataset ready:", audit["ready"])
print("Images:", sum(
    s["images"] for s in audit["splits"].values()
))
print("Training YAML:", DATA_YAML)
print("Training preparation: PASS")

In [ ]:
from pathlib import Path

CLEAN = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/"
    "fire_smoke_v14/"
    "polygon_review_20261110T141343824922Z/"
    "cleaned_detection_dataset"
)

print("Clean folder exists:", CLEAN.is_dir())

if CLEAN.is_dir():
    print("\nFiles and folders:")
    for item in sorted(CLEAN.iterdir()):
        print("📁" if item.is_dir() else "📄", item.name)

    print("\nYAML files:")
    for item in CLEAN.rglob("*.yaml"):
        print(item)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14")

print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())
print("Base folder exists:", BASE.is_dir())

if BASE.is_dir():
    print("\nReview folders:")
    for folder in sorted(BASE.glob("polygon_review_*")):
        print(folder)

    print("\nSearching for cleaned dataset...")
    matches = [
        p for p in BASE.rglob("cleaned_detection_dataset")
        if p.is_dir()
    ]

    for p in matches:
        print("FOUND:", p)
        print("Contents:", [x.name for x in p.iterdir()])

    print("\nTotal matches:", len(matches))
else:
    print("\nGoogle Drive may need to be remounted.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14"
)

CLEAN = (
    BASE
    / "polygon_review_20261110T141343824922Z"
    / "cleaned_detection_dataset"
)

print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())
print("Clean Dataset exists:", CLEAN.is_dir())

if CLEAN.is_dir():
    print("\nDataset contents:")
    for item in sorted(CLEAN.iterdir()):
        print(item.name)

    print("\nYAML files:")
    for file in CLEAN.rglob("*.yaml"):
        print(file)

In [ ]:
from pathlib import Path
import shutil
import sys
import torch

ROOT = Path("/content/yolo-multi-task-computer-vision")

CLEAN = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/"
    "fire_smoke_v14/"
    "polygon_review_20261110T141343824922Z/"
    "cleaned_detection_dataset"
)

LOCAL_DATA = ROOT / "datasets/fire_smoke_v14_training"

OUTPUT = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/"
    "fire_smoke_v14/training"
)

# Verify prerequisites
assert torch.cuda.is_available(), "Enable T4 GPU first"
assert ROOT.is_dir(), "Project repository is missing"
assert (CLEAN / "data.yaml").is_file()

sys.path.insert(0, str(ROOT))

# Copy clean dataset to Colab local storage
if not LOCAL_DATA.exists():
    shutil.copytree(CLEAN, LOCAL_DATA)
    print("Dataset copied successfully")
else:
    print("Local dataset already exists")

# Audit the dataset
from src.fire_smoke import audit_dataset, write_training_yaml

audit = audit_dataset(LOCAL_DATA)

assert audit["ready"] is True
assert len(audit["issues"]) == 0

# Generate training YAML
OUTPUT.mkdir(parents=True, exist_ok=True)

DATA_YAML = write_training_yaml(
    audit,
    OUTPUT / "data.training.yaml"
)

print("\n========== TRAINING PREPARATION ==========")
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset ready:", audit["ready"])
print("Audit issues:", len(audit["issues"]))
print("Total images:", sum(
    split["images"] for split in audit["splits"].values()
))
print("Training YAML:", DATA_YAML)
print("\nTraining preparation: PASS")

In [ ]:
from pathlib import Path
import json

print("========== AUDIT DIAGNOSIS ==========")

print("Dataset path:", LOCAL_DATA)
print("Dataset exists:", LOCAL_DATA.is_dir())
print("Audit ready:", audit["ready"])
print("Total issues:", len(audit["issues"]))

print("\n========== DATA.YAML ==========")
yaml_file = LOCAL_DATA / "data.yaml"

if yaml_file.is_file():
    print(yaml_file.read_text(encoding="utf-8"))
else:
    print("data.yaml NOT FOUND")

print("\n========== SPLIT SUMMARY ==========")
print(json.dumps(audit["splits"], indent=2)[:4000])

print("\n========== FIRST 10 ISSUES ==========")
for issue in audit["issues"][:10]:
    print(issue)

In [ ]:
from pathlib import Path
import yaml
from src.fire_smoke import audit_dataset, write_training_yaml

# Local dataset only — leave Google Drive original unchanged
yaml_path = LOCAL_DATA / "data.yaml"

assert yaml_path.is_file()
assert (LOCAL_DATA / "train/images").is_dir()
assert (LOCAL_DATA / "valid/images").is_dir()
assert (LOCAL_DATA / "test/images").is_dir()

# Read existing YAML
config = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))

# Update paths to local dataset
config["train"] = str(LOCAL_DATA / "train/images")
config["val"] = str(LOCAL_DATA / "valid/images")
config["test"] = str(LOCAL_DATA / "test/images")

# Save local training copy configuration
yaml_path.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8"
)

print("Updated LOCAL data.yaml:")
print(yaml_path.read_text(encoding="utf-8"))

# Re-run full audit
audit = audit_dataset(LOCAL_DATA)

print("\n========== FINAL AUDIT ==========")
print("Ready:", audit["ready"])
print("Issues:", len(audit["issues"]))

for split, info in audit["splits"].items():
    print(f"{split}: {info['images']} images")

assert audit["ready"] is True
assert len(audit["issues"]) == 0

# Generate training configuration
OUTPUT.mkdir(parents=True, exist_ok=True)

DATA_YAML = write_training_yaml(
    audit,
    OUTPUT / "data.training.yaml"
)

print("\nGPU:", torch.cuda.get_device_name(0))
print("Training YAML:", DATA_YAML)
print("TRAINING PREPARATION: PASS")

In [ ]:

from ultralytics import YOLO
from datetime import datetime, timezone
from pathlib import Path
import torch

# Safety checks
assert torch.cuda.is_available(), "GPU is unavailable"
assert audit["ready"] is True
assert len(audit["issues"]) == 0
assert Path(DATA_YAML).is_file()

# Unique run name to preserve previous experiments
RUN_NAME = (
    "fire_smoke_v14_seed42_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)

RUN_DIR = OUTPUT / RUN_NAME
assert not RUN_DIR.exists()

print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", DATA_YAML)
print("Run:", RUN_NAME)

# Load pretrained nano detection model
model = YOLO("yolo26n.pt", task="detect")

# Start training
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    deterministic=True,
    patience=0,
    workers=0,
    project=str(OUTPUT),
    name=RUN_NAME,
    exist_ok=False,
    save=True,
    save_period=10,
    plots=True,
    pretrained=True
)

# Confirm actual results
RUN_DIR = Path(model.trainer.save_dir)

print("\nTRAINING FINISHED")
print("Run directory:", RUN_DIR)
print("Best model:", RUN_DIR / "weights" / "best.pt")
print("Last model:", RUN_DIR / "weights" / "last.pt")
print("Best exists:", (RUN_DIR / "weights" / "best.pt").is_file())


In [ ]:

from pathlib import Path
from ultralytics import YOLO
import json
import torch

RUN_DIR = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14/"
    "training/fire_smoke_v14_seed42_20261114T125518Z"
)

BEST_PT = RUN_DIR / "weights/best.pt"

assert BEST_PT.is_file(), "best.pt not found"
assert Path(DATA_YAML).is_file()
assert torch.cuda.is_available()

model = YOLO(str(BEST_PT))

test_results = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
    batch=16,
    plots=True,
    project=str(RUN_DIR),
    name="test_evaluation",
    exist_ok=False
)

metrics = {
    "split": "test",
    "model": str(BEST_PT),
    "precision": float(test_results.box.mp),
    "recall": float(test_results.box.mr),
    "mAP50": float(test_results.box.map50),
    "mAP50_95": float(test_results.box.map),
}

report_path = RUN_DIR / "test_metrics.json"

report_path.write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

print("\n===== FINAL TEST RESULTS =====")
print(json.dumps(metrics, indent=2))
print("Saved report:", report_path)


In [ ]:

from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, Image
import random

RUN_DIR = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14/"
    "training/fire_smoke_v14_seed42_20261114T125518Z"
)

BEST_PT = RUN_DIR / "weights/best.pt"
TEST_IMAGES = LOCAL_DATA / "test/images"

assert BEST_PT.is_file()
assert TEST_IMAGES.is_dir()

images = [
    p for p in TEST_IMAGES.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

assert len(images) >= 4

random.seed(42)
sample_images = random.sample(images, 4)

model = YOLO(str(BEST_PT))

results = model.predict(
    source=[str(p) for p in sample_images],
    conf=0.25,
    imgsz=640,
    device=0,
    save=True,
    project=str(RUN_DIR),
    name="sample_predictions",
    exist_ok=False
)

print("\nSAMPLE PREDICTIONS")

for result in results:
    print(
        Path(result.path).name,
        "Detections:",
        len(result.boxes)
    )

    display(
        Image(data=__import__("cv2").imencode(
            ".jpg", result.plot()
        )[1].tobytes())
    )

print("\nPredictions saved!")


In [ ]:
from pathlib import Path
import requests

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14"
)

VIDEO_DIR = BASE / "demo_videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

source = (
    "https://raw.githubusercontent.com/"
    "punpayut/Fire-Detection/main/"
)

for filename in ["small_fire.mp4", "fire.mp4", "smoke.mp4"]:
    url = source + filename
    destination = VIDEO_DIR / filename

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    destination.write_bytes(response.content)

    print(
        filename,
        "Downloaded:",
        round(destination.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\nVIDEO DOWNLOAD COMPLETE")

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14"
)

VIDEO_DIR = BASE / "demo_videos"

RUN_DIR = (
    BASE / "training"
    / "fire_smoke_v14_seed42_20261114T125518Z"
)

BEST_PT = RUN_DIR / "weights/best.pt"

# Choose one of the downloaded videos
VIDEO_NAME = "small_fire.mp4"
VIDEO_PATH = VIDEO_DIR / VIDEO_NAME

assert BEST_PT.is_file(), "Trained model not found"
assert VIDEO_PATH.is_file(), "Video missing: run download cell first"
assert VIDEO_PATH.stat().st_size > 0, "Video is empty"

print("Model:", BEST_PT.name)
print("Video:", VIDEO_NAME)
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

model = YOLO(str(BEST_PT), task="detect")

# Run inference once and consume the video stream
results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.25,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    stream=True,
    project=str(RUN_DIR),
    name="video_small_fire",
    exist_ok=False,
    verbose=False
)

frames = 0
detected_frames = 0
fire_detections = 0
smoke_detections = 0

for result in results:
    frames += 1

    if len(result.boxes) > 0:
        detected_frames += 1

    for cls in result.boxes.cls.tolist():
        label = result.names[int(cls)]
        if label == "fire":
            fire_detections += 1
        elif label == "smoke":
            smoke_detections += 1

OUTPUT_DIR = RUN_DIR / "video_small_fire"

print("\n===== VIDEO INFERENCE RESULTS =====")
print("Processed frames:", frames)
print("Frames with detections:", detected_frames)
print("Fire detections:", fire_detections)
print("Smoke detections:", smoke_detections)
print("Saved output:", OUTPUT_DIR)

videos = [
    p for p in OUTPUT_DIR.iterdir()
    if p.suffix.lower() in {".mp4", ".avi", ".mov"}
]

print("Output files:", videos)

assert frames > 0, "No video frames were processed"
assert videos, "No output video was saved"

print("\nVIDEO INFERENCE: PASS")

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

AVI_PATH = (
    RUN_DIR / "video_small_fire" / "small_fire.avi"
)

MP4_PATH = AVI_PATH.with_suffix(".mp4")

assert AVI_PATH.is_file(), "AVI video not found"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(AVI_PATH),
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-an",
        str(MP4_PATH)
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

assert MP4_PATH.is_file()
assert MP4_PATH.stat().st_size > 0

print("MP4 saved:", MP4_PATH)

display(Video(str(MP4_PATH), embed=True))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14"
)

RUN_DIR = BASE / (
    "training/fire_smoke_v14_seed42_20261114T125518Z"
)

CHECKPOINT = BASE / "TASK1_PROGRESS_CHECKPOINT.json"

required_files = {
    "best_model": RUN_DIR / "weights/best.pt",
    "last_model": RUN_DIR / "weights/last.pt",
    "test_metrics": RUN_DIR / "test_metrics.json",
    "demo_video": RUN_DIR / "video_small_fire/small_fire.mp4",
}

print("===== SAVED ARTIFACTS =====")

for name, path in required_files.items():
    exists = path.is_file()
    print(f"{name}: {'OK' if exists else 'MISSING'}")
    assert exists, f"Missing: {path}"

test_metrics = json.loads(
    required_files["test_metrics"].read_text(encoding="utf-8")
)

progress = {
    "project": "YOLO Multi-Task Computer Vision",
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "task_1": {
        "name": "Fire & Smoke Detection",
        "status": "practical_work_completed",
        "dataset_images": 5596,
        "audit_issues": 0,
        "model": "yolo26n.pt",
        "epochs_completed": 50,
        "test_metrics": test_metrics,
        "artifacts": {
            key: str(path)
            for key, path in required_files.items()
        },
        "video_frames_processed": 384,
        "video_frames_with_detections": 314,
        "fire_detections": 428,
        "smoke_detections": 155,
        "remaining": [
            "Finalize Task 1 documentation",
            "Review video licensing before publishing"
        ]
    },
    "next_task": {
        "name": "Task 2 - Object Counting",
        "status": "not_started"
    }
}

CHECKPOINT.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

assert CHECKPOINT.is_file()

print("\nPROJECT CHECKPOINT SAVED SUCCESSFULLY")
print("Location:", CHECKPOINT)

## 9. Training — completed and evaluated

The Fire & Smoke YOLO26n detection model was trained for 50 epochs
using the cleaned Roboflow version 14 dataset.

- Dataset: 5,596 images; 2 classes (fire, smoke)
- Training: 3,940 images
- Validation: 1,017 images
- Test: 639 images
- GPU: Tesla T4
- Image size: 640
- Seed: 42
- Checkpoints: `best.pt` and `last.pt`

### Held-out test results

| Metric | Result |
|---|---:|
| Precision | 70.86% |
| Recall | 54.73% |
| mAP@50 | 63.02% |
| mAP@50-95 | 28.00% |

Image inference and video inference were executed successfully.
The demonstration video contained 384 processed frames, with
314 frames containing detections.

The lower smoke recall is an important limitation.
These results do not qualify the model as a standalone fire alarm.

The original dataset, reviewed corrections, trained weights,
metrics and video outputs are preserved on Google Drive.
Do not rerun training solely to reproduce this report.


## Review saved training and test artifacts

The following cell reads existing results without retraining.


In [ ]:
from pathlib import Path
import json

FIRE_BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/fire_smoke_v14"
)
FIRE_CLEAN = (
    FIRE_BASE
    / "polygon_review_20261110T141343824922Z"
    / "cleaned_detection_dataset"
)
FIRE_RUN = (
    FIRE_BASE
    / "training"
    / "fire_smoke_v14_seed42_20261114T125518Z"
)

FIRE_BEST = FIRE_RUN / "weights" / "best.pt"
FIRE_REPORT = FIRE_RUN / "test_metrics.json"

assert (FIRE_CLEAN / "data.yaml").is_file()
assert FIRE_BEST.is_file()
assert FIRE_REPORT.is_file()

fire_metrics = json.loads(
    FIRE_REPORT.read_text(encoding="utf-8")
)

print("FIRE & SMOKE — SAVED TEST RESULTS")
for field in [
    "precision", "recall", "mAP50", "mAP50_95"
]:
    print(field, ":", fire_metrics.get(field))

print("FIRE SAVED ARTIFACT RESTORE: PASS")
